# Vetores, strings e passagem por referência — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial retoma o programa `agenda.c` visto em aula e segue de onde a aula parou: as
**tarefas de modificação** e o **desafio**. Antes delas há três blocos de prática, um por tópico.

## Objetivos

Ao final deste tutorial você será capaz de:

- Explicar por que `sizeof` devolve o tamanho do vetor em `main` e o tamanho de um ponteiro dentro
  de uma função, e por isso passar o tamanho como parâmetro;
- Escrever funções que percorrem e alteram strings usando o terminador `'\0'`;
- Distinguir passagem por valor de passagem do endereço, e usar ponteiros como parâmetros de saída;
- Usar `const` para declarar que uma função só lê o dado que recebeu.

> Cada célula `%%writefile` grava um arquivo `.c` no diretório do notebook; a célula seguinte
> compila e executa. Rode as células **na ordem**.

In [ ]:
# Confira se o gcc está disponível no seu ambiente
!gcc --version | head -1

## O programa da aula

Recompile e execute o programa condutor **exatamente como saiu da aula** e confira que a saída
bate com a que vimos no slide.

In [ ]:
%%writefile agenda.c
#include <stdio.h>

#define MAX_CONTATOS 4
#define MAX_NOME     20

void medir(int idades[], int n) {
    printf("A: sizeof dentro da funcao = %zu | n = %d\n", sizeof(idades), n);
}

void envelhecer(int idades[], int n) {
    for (int i = 0; i < n; i++)
        idades[i] = idades[i] + 1;
}

int meu_tamanho(const char *s) {
    int n = 0;
    while (s[n] != '\0')
        n++;
    return n;
}

void maiusculas(char *s) {
    for (int i = 0; s[i] != '\0'; i++)
        if (s[i] >= 'a' && s[i] <= 'z')
            s[i] = s[i] - 'a' + 'A';
}

void estatisticas(int idades[], int n, int *soma, int *maior) {
    *soma  = 0;
    *maior = idades[0];
    for (int i = 0; i < n; i++) {
        *soma = *soma + idades[i];
        if (idades[i] > *maior)
            *maior = idades[i];
    }
}

int main(void) {
    char nomes[MAX_CONTATOS][MAX_NOME] = {"ana", "bruno", "carla", "daniel"};
    int idades[MAX_CONTATOS] = {21, 19, 23, 20};
    int soma, maior;
    printf("A: sizeof em main = %zu\n", sizeof(idades));
    medir(idades, MAX_CONTATOS);
    envelhecer(idades, MAX_CONTATOS);
    printf("B: idades = %d %d %d %d\n", idades[0], idades[1], idades[2], idades[3]);
    printf("C: sizeof(nomes[1]) = %zu | meu_tamanho = %d\n",
           sizeof(nomes[1]), meu_tamanho(nomes[1]));
    maiusculas(nomes[1]);
    printf("D: nomes[1] = %s\n", nomes[1]);
    estatisticas(idades, MAX_CONTATOS, &soma, &maior);
    printf("E: soma = %d | maior = %d\n", soma, maior);
    return 0;
}

In [ ]:
# O aviso do -Wall sobre "sizeof on array function parameter" é ESPERADO:
# é o compilador descrevendo exatamente o fenômeno que a aula investiga.
!gcc -Wall agenda.c -o agenda && ./agenda
!./agenda | grep -q 'E: soma = 87 | maior = 24' && echo OK || echo 'Verifique: esperava soma = 87 e maior = 24' 

## 1. O tamanho que se perde

Quando um vetor é passado para uma função, o nome do vetor **decai** no endereço do seu primeiro
elemento. A função não recebe uma cópia dos elementos: recebe um ponteiro.

Consequências:

- `sizeof` dentro da função mede o ponteiro (8 bytes nesta máquina), não o vetor;
- `int v[]`, `int v[4]` e `int *v` são o **mesmo** parâmetro para o compilador;
- o tamanho precisa viajar num parâmetro à parte;
- escrever em `v[i]` dentro da função altera o vetor de quem chamou.

In [ ]:
%%writefile decaimento.c
#include <stdio.h>

/* as tres assinaturas abaixo sao equivalentes para o compilador */
void por_colchetes(int v[], int n)   { printf("  v[1]        = %d\n", v[1]); }
void por_tamanho(int v[4], int n)    { printf("  v[1]        = %d\n", v[1]); }
void por_ponteiro(int *v, int n)     { printf("  *(v + 1)    = %d\n", *(v + 1)); }

void tenta_medir(int v[], int n) {
    printf("  sizeof(v) dentro     = %zu  (tamanho de um int *)\n", sizeof(v));
    printf("  sizeof(v)/sizeof(v[0]) = %zu  <-- conta ERRADA aqui dentro\n",
           sizeof(v) / sizeof(v[0]));
}

int main(void) {
    int v[4] = {10, 20, 30, 40};

    printf("Em main:\n");
    printf("  sizeof(v) = %zu | elementos = %zu\n", sizeof(v), sizeof(v) / sizeof(v[0]));
    printf("  v == &v[0]? %s\n", (v == &v[0]) ? "sim" : "nao");

    printf("Dentro da funcao:\n");
    tenta_medir(v, 4);

    printf("Mesmo parametro, tres escritas:\n");
    por_colchetes(v, 4);
    por_tamanho(v, 4);
    por_ponteiro(v, 4);
    return 0;
}

In [ ]:
# Os avisos do -Wall sobre sizeof aqui são esperados: é o compilador
# apontando a conta que só funciona onde o vetor foi declarado.
!gcc -Wall decaimento.c -o decaimento && ./decaimento

In [ ]:
%%writefile ex1.c
#include <stdio.h>

/* Exercicio 1: some os elementos de um vetor.
   TODO: complete a funcao. Ela recebe o vetor e o tamanho porque, aqui dentro,
   nao existe como descobrir quantos elementos ha.
   Use const: a funcao so' le o vetor. */
int soma_vetor(const int v[], int n) {
    /* TODO: implemente aqui */
    return 0;
}

int main(void) {
    int a[5] = {1, 2, 3, 4, 5};
    int b[3] = {10, -2, 7};
    printf("%d %d\n", soma_vetor(a, 5), soma_vetor(b, 3));
    return 0;
}

In [ ]:
!gcc -Wall ex1.c -o ex1 && ./ex1
!./ex1 | grep -q '^15 15$' && echo OK || echo 'Verifique: esperava "15 15"' 

## 2. Strings: um vetor de `char` que termina em `'\0'`

Uma string em C é um vetor de `char` cujo fim é marcado pelo byte `'\0'`. Nada mais guarda o
comprimento: quem quiser saber onde a palavra acaba precisa **andar** até o terminador.

- `sizeof(s)` — a capacidade declarada (só funciona onde `s` foi declarado como vetor);
- `strlen(s)` — o conteúdo atual, sem contar o `'\0'`;
- `char a[20] = "ana"` cria um vetor **seu**, que pode ser alterado;
- `const char *b = "ana"` aponta para um literal que **não** pode ser alterado;
- comparação de strings é com `strcmp`, nunca com `==`.

In [ ]:
%%writefile strings.c
#include <stdio.h>
#include <string.h>

int main(void) {
    char nome[20] = "bruno";

    printf("sizeof(nome) = %zu (capacidade) | strlen(nome) = %zu (conteudo)\n",
           sizeof(nome), strlen(nome));

    /* onde esta' o terminador? */
    size_t fim = strlen(nome);
    for (int i = 0; i < 7; i++)
        printf("nome[%d] = %3d  %s\n", i, nome[i],
               ((size_t) i == fim) ? "<-- '\\0': fim da string" :
               ((size_t) i <  fim) ? "" : "resto do vetor: nao faz parte da string");

    /* == compara enderecos; strcmp compara conteudo.
       O compilador avisa "array comparison always evaluates to false":
       o aviso E' a licao -- nunca compare strings com ==. */
    char outro[20] = "bruno";
    printf("nome == outro ?      %s\n", (nome == outro) ? "sim" : "nao");
    printf("strcmp(nome, outro)? %s\n", (strcmp(nome, outro) == 0) ? "iguais" : "diferentes");

    /* apagar o terminador no meio encurta a string */
    nome[3] = '\0';
    printf("depois de nome[3] = '\\0': \"%s\" (strlen = %zu)\n", nome, strlen(nome));
    return 0;
}

In [ ]:
!gcc -Wall strings.c -o strings && ./strings

In [ ]:
%%writefile ex2.c
#include <stdio.h>

/* Exercicio 2: conte as vogais de uma string.
   TODO: percorra s ate' o '\0' e conte a, e, i, o, u (minusculas).
   Note o const: esta funcao nao deve alterar a string recebida. */
int conta_vogais(const char *s) {
    /* TODO: implemente aqui */
    return 0;
}

int main(void) {
    printf("%d %d %d\n", conta_vogais("programacao"), conta_vogais("xyz"), conta_vogais(""));
    return 0;
}

In [ ]:
!gcc -Wall ex2.c -o ex2 && ./ex2
!./ex2 | grep -q '^5 0 0$' && echo OK || echo 'Verifique: esperava "5 0 0"' 

## 3. Passagem por valor e por referência

C tem **um único** mecanismo: a função sempre recebe uma cópia do argumento. Quando o argumento é
um endereço, a cópia do endereço aponta para o mesmo lugar — e alterar `*p` altera o original.

| Quero que a função…       | Parâmetro                 | Na chamada        |
|---------------------------|---------------------------|-------------------|
| só leia um número         | `int x`                   | `f(x)`            |
| altere um número          | `int *x`                  | `f(&x)`           |
| só leia um vetor/string   | `const T *v, int n`       | `f(v, n)`         |
| altere um vetor/string    | `T *v, int n`             | `f(v, n)`         |

Vetores e strings não levam `&` na chamada porque já decaem em endereço.

In [ ]:
%%writefile passagem.c
#include <stdio.h>

void nao_muda(int x)  { x = 99; }        /* recebe uma copia          */
void muda(int *x)     { *x = 99; }       /* recebe o endereco         */

/* dois resultados de uma vez, por parametros de saida */
void min_max(const int v[], int n, int *menor, int *maior) {
    *menor = v[0];
    *maior = v[0];
    for (int i = 1; i < n; i++) {
        if (v[i] < *menor) *menor = v[i];
        if (v[i] > *maior) *maior = v[i];
    }
}

int main(void) {
    int a = 1;
    nao_muda(a);  printf("depois de nao_muda(a): a = %d\n", a);
    muda(&a);     printf("depois de muda(&a):    a = %d\n", a);

    int v[6] = {7, 2, 9, 4, 9, -1};
    int menor, maior;
    min_max(v, 6, &menor, &maior);
    printf("menor = %d | maior = %d\n", menor, maior);
    return 0;
}

In [ ]:
!gcc -Wall passagem.c -o passagem && ./passagem

In [ ]:
%%writefile ex3.c
#include <stdio.h>

/* Exercicio 3: divisao inteira devolvendo quociente E resto.
   TODO: escreva em *q e *r. A funcao devolve 1 se deu certo e 0 se b == 0
   (assim o codigo de erro vai no return e os resultados vao pelos ponteiros). */
int divide(int a, int b, int *q, int *r) {
    /* TODO: implemente aqui */
    return 0;
}

int main(void) {
    int q, r;
    if (divide(17, 5, &q, &r))
        printf("17/5 -> q=%d r=%d\n", q, r);
    printf("divide(1, 0, &q, &r) = %d\n", divide(1, 0, &q, &r));
    return 0;
}

In [ ]:
!gcc -Wall ex3.c -o ex3 && ./ex3
!./ex3 | grep -q 'q=3 r=2' && echo OK || echo 'Verifique: esperava q=3 r=2' 

## Tarefas da aula

As três tarefas propostas no slide **Altere o programa**, agora num único arquivo com testes.

1. `minusculas(char *s)` — a inversa de `maiusculas`.
2. `media(const int v[], int n, float *m)` — a média pelo parâmetro de saída.
   Por que `const` cabe aqui e não em `envelhecer`?
3. `busca(char nomes[][MAX_NOME], int n, const char *alvo)` — índice do contato ou `-1`.

In [ ]:
%%writefile tarefas.c
#include <stdio.h>
#include <string.h>

#define MAX_CONTATOS 4
#define MAX_NOME     20

/* Tarefa 1 */
void minusculas(char *s) {
    /* TODO: troque cada letra maiuscula pela minuscula correspondente */
}

/* Tarefa 2 */
void media(const int v[], int n, float *m) {
    /* TODO: escreva a media em *m (cuidado com a divisao inteira) */
}

/* Tarefa 3 */
int busca(char nomes[][MAX_NOME], int n, const char *alvo) {
    /* TODO: devolva o indice de alvo em nomes, ou -1 se nao houver */
    return -1;
}

int main(void) {
    char nomes[MAX_CONTATOS][MAX_NOME] = {"ana", "bruno", "carla", "daniel"};
    int idades[MAX_CONTATOS] = {22, 20, 24, 21};

    char nome[MAX_NOME] = "DANIEL";
    minusculas(nome);
    printf("1: %s\n", nome);

    float m;
    media(idades, MAX_CONTATOS, &m);
    printf("2: media = %.2f\n", m);

    printf("3: carla=%d zeca=%d\n",
           busca(nomes, MAX_CONTATOS, "carla"), busca(nomes, MAX_CONTATOS, "zeca"));
    return 0;
}

In [ ]:
!gcc -Wall tarefas.c -o tarefas && ./tarefas
!./tarefas | grep -q '^1: daniel$' && echo 'Tarefa 1 OK' || echo 'Tarefa 1: esperava "daniel"'
!./tarefas | grep -q '^2: media = 21.75$' && echo 'Tarefa 2 OK' || echo 'Tarefa 2: esperava 21.75'
!./tarefas | grep -q '^3: carla=2 zeca=-1$' && echo 'Tarefa 3 OK' || echo 'Tarefa 3: esperava carla=2 zeca=-1' 

## Desafio: agenda ordenada

Escreva `ordena_nomes(char nomes[][MAX_NOME], int idades[], int n)` que ordene os contatos em ordem
alfabética **mantendo cada idade junto do seu nome**.

Requisitos:

- compare nomes com `strcmp` e troque strings com `strcpy` e um vetor auxiliar
  (não existe `nomes[i] = nomes[j]` para strings);
- toda troca de nome deve vir acompanhada da troca da idade correspondente;
- a função não devolve nada — o efeito aparece nos vetores de `main`.

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <string.h>

#define MAX_CONTATOS 4
#define MAX_NOME     20

void ordena_nomes(char nomes[][MAX_NOME], int idades[], int n) {
    /* TODO: ordene nomes em ordem alfabetica, levando idades junto.
       Dica: um bubble sort basta. Para trocar duas strings use um char aux[MAX_NOME]
       e tres strcpy; para trocar duas idades use um int auxiliar. */
}

int main(void) {
    char nomes[MAX_CONTATOS][MAX_NOME] = {"daniel", "ana", "carla", "bruno"};
    int idades[MAX_CONTATOS] = {21, 22, 24, 20};

    ordena_nomes(nomes, idades, MAX_CONTATOS);
    for (int i = 0; i < MAX_CONTATOS; i++)
        printf("%s %d\n", nomes[i], idades[i]);
    return 0;
}

In [ ]:
!gcc -Wall desafio.c -o desafio && ./desafio
!./desafio | tr '\n' '|' | grep -q '^ana 22|bruno 20|carla 24|daniel 21|$' && echo 'Desafio OK' || echo 'Verifique: esperava ana 22, bruno 20, carla 24, daniel 21 (nesta ordem)' 

## Para ir além

- Reescreva `conta_vogais` usando aritmética de ponteiros (`while (*s) { ... s++; }`) em vez de
  índices. O resultado deve ser o mesmo.
- Tente compilar uma função que recebe `const char *s` e escreve em `s[0]`. Leia a mensagem de erro:
  é o compilador cobrando o contrato que você assinou.
- Na próxima aula: e quando o tamanho do vetor só é conhecido em tempo de execução? Aí o ponteiro
  deixa de apontar para algo que já existe e passa a apontar para memória que você mesmo reserva.

## Referências

Veja o arquivo `../referencias.bib` para a lista completa.